# F1 Race Strategy Simulator and Predictor

This notebook uses four CSV files: `races.csv`, `results.csv`, `drivers.csv`, and `constructors.csv`.

**What it does**
1. Loads and checks the datasets.
2. Cleans missing values and merges the four tables.
3. Performs exploratory data analysis (EDA) with charts.
4. Creates historical driver/team features using only earlier races.
5. Trains models to predict finishing position and podium probability.
6. Provides a simple what-if simulator for a selected driver, team, and grid position.

> This is a historical baseline. The four files do not contain tyre compounds, tyre wear, fuel loads, pit-stop timing, or lap-by-lap data, so the simulator cannot yet optimize real pit-stop strategies.


## 1. Import libraries

Run this cell first. Google Colab already includes most of these libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, classification_report
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

print("Libraries loaded successfully.")

## 2. Upload the four CSV files

In Colab, click the folder icon on the left, choose **Upload**, and upload all four CSV files. Keep the filenames the same as below.

If the files are already in your Google Drive, you can mount Drive and change `DATA_FOLDER` to the folder containing them.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select constructors.csv, races.csv, results.csv, drivers.csv

DATA_FOLDER = ""  # Leave blank if files were uploaded directly to Colab

def file_path(filename):
    if DATA_FOLDER:
        return f"{DATA_FOLDER.rstrip('/')}/{filename}"
    return filename

print("Uploaded files:", list(uploaded.keys()))

## 3. Read the data and inspect it

The original files use `\\N` for some missing values. We convert those markers to real missing values (`NaN`).

In [ ]:
constructors = pd.read_csv(file_path("constructors.csv"), na_values=["\\N"])
races = pd.read_csv(file_path("races.csv"), na_values=["\\N"])
results = pd.read_csv(file_path("results.csv"), na_values=["\\N"])
drivers = pd.read_csv(file_path("drivers.csv"), na_values=["\\N"])

datasets = {
    "constructors": constructors,
    "races": races,
    "results": results,
    "drivers": drivers
}

for name, data in datasets.items():
    print(f"\\n{name.upper()}: {data.shape[0]} rows and {data.shape[1]} columns")
    display(data.head(3))

## 4. Check data quality

We check column types, duplicate rows, missing values, and key IDs before merging.

In [ ]:
for name, data in datasets.items():
    print(f"\\n--- {name.upper()} ---")
    print("Duplicate rows:", data.duplicated().sum())
    missing = data.isna().sum()
    print("Columns with missing values:")
    display(missing[missing > 0].sort_values(ascending=False).to_frame("missing_count").head(12))

print("Unique race IDs in races:", races["raceId"].nunique())
print("Unique result race IDs:", results["raceId"].nunique())
print("Unique driver IDs:", drivers["driverId"].nunique())
print("Unique constructor IDs:", constructors["constructorId"].nunique())

## 5. Merge all four datasets

The relationships are:
- `results.raceId` → `races.raceId`
- `results.driverId` → `drivers.driverId`
- `results.constructorId` → `constructors.constructorId`

We use left joins from `results` so every recorded race result is retained.

In [ ]:
# Keep only useful descriptive columns from the lookup tables.
race_columns = ["raceId", "year", "round", "circuitId", "name", "date"]
driver_columns = ["driverId", "driverRef", "code", "forename", "surname", "nationality"]
constructor_columns = ["constructorId", "constructorRef", "name", "nationality"]

race_info = races[race_columns].rename(columns={"name": "race_name"})
driver_info = drivers[driver_columns].rename(columns={"nationality": "driver_nationality"})
constructor_info = constructors[constructor_columns].rename(
    columns={"name": "constructor_name", "nationality": "constructor_nationality"}
)

df = results.merge(race_info, on="raceId", how="left")
df = df.merge(driver_info, on="driverId", how="left")
df = df.merge(constructor_info, on="constructorId", how="left")

# Create readable names for reports and charts.
df["driver_name"] = (df["forename"].fillna("").astype(str) + " " +
                     df["surname"].fillna("").astype(str)).str.strip()

# Convert important numeric columns to numeric values.
numeric_columns = [
    "year", "round", "grid", "positionOrder", "points", "laps",
    "milliseconds", "fastestLapSpeed", "fastestLap", "rank"
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values(["date", "raceId", "positionOrder"]).reset_index(drop=True)

print("Merged dataset shape:", df.shape)
display(df.head())
print("Years covered:", df["year"].min(), "to", df["year"].max())
print("Missing race names after merge:", df["race_name"].isna().sum())
print("Missing driver names after merge:", df["driver_name"].eq("").sum())

## 6. Save the merged dataset

This file can be downloaded and reused in the project.

In [ ]:
df.to_csv("f1_merged_dataset.csv", index=False)
print("Saved: f1_merged_dataset.csv")

from google.colab import files
files.download("f1_merged_dataset.csv")

## 7. Exploratory data analysis (EDA)

### 7.1 Number of races per year

In [ ]:
races_per_year = races.groupby("year")["raceId"].nunique()

plt.figure(figsize=(13, 5))
races_per_year.plot(kind="bar")
plt.title("Number of Formula 1 Races per Year")
plt.xlabel("Year")
plt.ylabel("Number of races")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

### 7.2 Most successful drivers by championship points

Points systems changed over time, so this chart shows recorded points rather than a perfectly normalized comparison across eras.

In [ ]:
driver_points = (
    df.groupby("driver_name", dropna=True)["points"]
      .sum()
      .sort_values(ascending=False)
      .head(15)
)

plt.figure(figsize=(11, 6))
driver_points.sort_values().plot(kind="barh")
plt.title("Top 15 Drivers by Total Recorded Points")
plt.xlabel("Total points")
plt.ylabel("Driver")
plt.tight_layout()
plt.show()

### 7.3 Constructor performance

In [ ]:
constructor_points = (
    df.groupby("constructor_name")["points"]
      .sum()
      .sort_values(ascending=False)
      .head(12)
)

plt.figure(figsize=(11, 6))
constructor_points.sort_values().plot(kind="barh")
plt.title("Top Constructors by Total Recorded Points")
plt.xlabel("Total points")
plt.ylabel("Constructor")
plt.tight_layout()
plt.show()

### 7.4 Starting grid versus finishing position

Lower position numbers are better. This plot helps show the relationship between starting position and final result.

In [ ]:
grid_plot = df.dropna(subset=["grid", "positionOrder"]).copy()
grid_plot = grid_plot[(grid_plot["grid"] > 0) & (grid_plot["positionOrder"] > 0)]

plt.figure(figsize=(9, 6))
sns.scatterplot(data=grid_plot.sample(min(5000, len(grid_plot)), random_state=42),
                x="grid", y="positionOrder", alpha=0.35)
plt.title("Starting Grid vs Finishing Position")
plt.xlabel("Starting grid position")
plt.ylabel("Finishing position (lower is better)")
plt.tight_layout()
plt.show()

print("Correlation between grid and finishing position:",
      round(grid_plot[["grid", "positionOrder"]].corr().iloc[0, 1], 3))

### 7.5 Finish status and race completion

The `statusId` field represents how a result ended. We show the most common statuses.

In [ ]:
status_counts = df["statusId"].value_counts().head(12)

plt.figure(figsize=(10, 5))
status_counts.sort_values().plot(kind="barh")
plt.title("Most Common Result Status IDs")
plt.xlabel("Number of results")
plt.ylabel("Status ID")
plt.tight_layout()
plt.show()

display(df[["statusId", "positionText", "positionOrder", "laps"]].head(10))

## 8. Prepare data for machine learning

We predict `positionOrder` (the result order recorded by the dataset). To avoid data leakage, we do **not** use final race points, final position text, fastest lap, or other information that would only be known after the race.

We create historical driver and constructor features by shifting their previous results before calculating expanding averages. That means a race's prediction only uses data from earlier races.

In [ ]:
model_df = df.copy()

# Keep rows with a usable target and starting grid.
model_df = model_df.dropna(subset=["positionOrder", "grid", "year", "raceId"]).copy()
model_df = model_df[model_df["grid"] > 0].copy()

# Sort chronologically before building previous-race features.
model_df = model_df.sort_values(["date", "raceId", "positionOrder"]).reset_index(drop=True)

# Previous performance for each driver: shift(1) prevents using the current race.
model_df["driver_previous_finish"] = (
    model_df.groupby("driverId")["positionOrder"].transform(lambda s: s.shift(1))
)
model_df["driver_avg_finish_before"] = (
    model_df.groupby("driverId")["positionOrder"]
    .transform(lambda s: s.shift(1).expanding().mean())
)
model_df["driver_avg_points_before"] = (
    model_df.groupby("driverId")["points"]
    .transform(lambda s: s.shift(1).expanding().mean())
)

# Previous performance for each constructor.
model_df["team_previous_finish"] = (
    model_df.groupby("constructorId")["positionOrder"].transform(lambda s: s.shift(1))
)
model_df["team_avg_finish_before"] = (
    model_df.groupby("constructorId")["positionOrder"]
    .transform(lambda s: s.shift(1).expanding().mean())
)
model_df["team_avg_points_before"] = (
    model_df.groupby("constructorId")["points"]
    .transform(lambda s: s.shift(1).expanding().mean())
)

# Replace missing history for drivers/teams at their first race.
# The imputer below will handle these missing feature values.
feature_columns = [
    "year", "round", "circuitId", "grid", "driverId", "constructorId",
    "driver_previous_finish", "driver_avg_finish_before",
    "driver_avg_points_before", "team_previous_finish",
    "team_avg_finish_before", "team_avg_points_before"
]

model_df = model_df.sort_values(["date", "raceId"]).reset_index(drop=True)

print("Rows available for modeling:", len(model_df))
display(model_df[feature_columns + ["positionOrder"]].head(10))

## 9. Train and evaluate the finishing-position predictor

We use a chronological split: older races for training and newer races for testing. This is more realistic than randomly mixing old and new races.

The model's output is an estimated finishing position. Since lower positions are better, compare errors as the average number of places off the recorded result.

In [ ]:
# Use the latest 20% of race years as a holdout test period.
years = sorted(model_df["year"].dropna().unique())
split_index = max(1, int(len(years) * 0.8))
test_start_year = years[split_index]

train_data = model_df[model_df["year"] < test_start_year].copy()
test_data = model_df[model_df["year"] >= test_start_year].copy()

X_train = train_data[feature_columns]
y_train = train_data["positionOrder"]
X_test = test_data[feature_columns]
y_test = test_data["positionOrder"]

regressor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])

regressor.fit(X_train, y_train)
position_predictions = regressor.predict(X_test)

mae = mean_absolute_error(y_test, position_predictions)
rmse = np.sqrt(mean_squared_error(y_test, position_predictions))

print("Training years:", int(train_data["year"].min()), "to", int(train_data["year"].max()))
print("Testing years:", int(test_data["year"].min()), "to", int(test_data["year"].max()))
print(f"Mean Absolute Error (positions): {mae:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")

comparison = test_data[["year", "race_name", "driver_name", "constructor_name", "grid", "positionOrder"]].copy()
comparison["predicted_position"] = np.round(position_predictions, 1)
comparison = comparison.rename(columns={"positionOrder": "actual_position"})
display(comparison.head(20))

## 10. Predict podium probability

A podium is defined here as a recorded finishing position from 1 to 3. We train a separate classifier and report its test accuracy and classification report. For an imbalanced target, also pay attention to precision and recall for the podium class.

In [ ]:
train_data = train_data.copy()
test_data = test_data.copy()
train_data["podium"] = train_data["positionOrder"].between(1, 3).astype(int)
test_data["podium"] = test_data["positionOrder"].between(1, 3).astype(int)

podium_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=250,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

podium_model.fit(train_data[feature_columns], train_data["podium"])
podium_predictions = podium_model.predict(test_data[feature_columns])
podium_probabilities = podium_model.predict_proba(test_data[feature_columns])[:, 1]

print("Podium classification accuracy:", round(accuracy_score(test_data["podium"], podium_predictions), 3))
print(classification_report(test_data["podium"], podium_predictions,
                            target_names=["Not podium", "Podium"], zero_division=0))

podium_comparison = test_data[["year", "race_name", "driver_name", "constructor_name", "grid", "podium"]].copy()
podium_comparison["predicted_podium_probability"] = np.round(podium_probabilities, 3)
display(podium_comparison.sort_values("predicted_podium_probability", ascending=False).head(15))

## 10A. Save the trained models as `.pkl` files

Run this cell **after Section 10**, when both `regressor` and `podium_model` have been trained. It saves the complete scikit-learn pipelines, including their missing-value imputers, so they can be loaded later without retraining. It also saves the feature-column order needed for predictions.


In [ ]:
import pickle
from google.colab import files

# Save the finishing-position model (regression pipeline).
with open("f1_position_model.pkl", "wb") as f:
    pickle.dump(regressor, f)

# Save the podium probability model (classification pipeline).
with open("f1_podium_model.pkl", "wb") as f:
    pickle.dump(podium_model, f)

# Save feature names/order and useful metadata for loading the models later.
model_metadata = {
    "feature_columns": feature_columns,
    "target_position": "positionOrder",
    "podium_definition": "positionOrder between 1 and 3",
    "training_year_start": int(train_data["year"].min()),
    "training_year_end": int(train_data["year"].max()),
    "test_year_start": int(test_data["year"].min()),
    "test_year_end": int(test_data["year"].max()),
}
with open("f1_model_metadata.pkl", "wb") as f:
    pickle.dump(model_metadata, f)

print("Saved files:")
print("- f1_position_model.pkl")
print("- f1_podium_model.pkl")
print("- f1_model_metadata.pkl")

# Download the three files to your computer.
files.download("f1_position_model.pkl")
files.download("f1_podium_model.pkl")
files.download("f1_model_metadata.pkl")


### Load the saved models later

To load these files in another Python environment, place the three `.pkl` files in the project folder and run:

```python
import pickle

with open("f1_position_model.pkl", "rb") as f:
    regressor = pickle.load(f)

with open("f1_podium_model.pkl", "rb") as f:
    podium_model = pickle.load(f)

with open("f1_model_metadata.pkl", "rb") as f:
    model_metadata = pickle.load(f)

feature_columns = model_metadata["feature_columns"]
```

**Important:** Use the same feature engineering and feature order used during training. Only load `.pkl` files that you created or trust, because loading pickle files from unknown sources can execute unsafe code. Keep the scikit-learn/Python environment compatible with the one used to save them.


## 11. See which features influenced the model

Feature importance is a rough indicator of which inputs the random forest used most. It does not prove causation.

In [ ]:
importance = pd.Series(
    regressor.named_steps["model"].feature_importances_,
    index=feature_columns
).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importance.plot(kind="barh")
plt.title("Feature Importance for Finishing Position Model")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 12. Build a simple race strategy simulator

This function lets you choose a driver, constructor, year, round, and starting grid position. It uses historical averages before the selected season/round to estimate driver and team form, then passes those features to the trained models.

This is a **what-if finishing outcome simulator**, not a tyre/pit-stop optimizer. The model uses the selected grid position as an input, so changing grid position lets you compare hypothetical starting scenarios.

In [ ]:
def simulate_race(driver_name, constructor_name, year, round_number, grid_position):
    # Find the latest historical rows strictly before the requested race.
    earlier = model_df[
        (model_df["year"] < year) |
        ((model_df["year"] == year) & (model_df["round"] < round_number))
    ].copy()

    driver_history = earlier[earlier["driver_name"].str.lower() == driver_name.lower()]
    team_history = earlier[earlier["constructor_name"].str.lower() == constructor_name.lower()]

    # Use previous records where available. Otherwise the model imputer fills missing history.
    driver_previous_finish = driver_history["positionOrder"].iloc[-1] if len(driver_history) else np.nan
    driver_avg_finish = driver_history["positionOrder"].mean() if len(driver_history) else np.nan
    driver_avg_points = driver_history["points"].mean() if len(driver_history) else np.nan

    team_previous_finish = team_history["positionOrder"].iloc[-1] if len(team_history) else np.nan
    team_avg_finish = team_history["positionOrder"].mean() if len(team_history) else np.nan
    team_avg_points = team_history["points"].mean() if len(team_history) else np.nan

    # Use a circuit ID from the selected season/round when available.
    race_match = races[(races["year"] == year) & (races["round"] == round_number)]
    circuit_id = race_match["circuitId"].iloc[0] if len(race_match) else np.nan

    input_row = pd.DataFrame([{
        "year": year,
        "round": round_number,
        "circuitId": circuit_id,
        "grid": grid_position,
        "driverId": driver_history["driverId"].iloc[-1] if len(driver_history) else np.nan,
        "constructorId": team_history["constructorId"].iloc[-1] if len(team_history) else np.nan,
        "driver_previous_finish": driver_previous_finish,
        "driver_avg_finish_before": driver_avg_finish,
        "driver_avg_points_before": driver_avg_points,
        "team_previous_finish": team_previous_finish,
        "team_avg_finish_before": team_avg_finish,
        "team_avg_points_before": team_avg_points
    }])[feature_columns]

    predicted_position = float(regressor.predict(input_row)[0])
    podium_probability = float(podium_model.predict_proba(input_row)[0, 1])

    return {
        "Driver": driver_name,
        "Constructor": constructor_name,
        "Year": year,
        "Round": round_number,
        "Hypothetical grid position": grid_position,
        "Predicted finishing position": round(predicted_position, 1),
        "Estimated podium probability": f"{podium_probability:.1%}"
    }

# Example: replace these names with entries from your dataset.
print("Example drivers:", sorted(df["driver_name"].dropna().unique())[:15])
print("Example constructors:", sorted(df["constructor_name"].dropna().unique())[:15])

## 13. Run a simulation

Edit the values below using driver and constructor names from the printed lists. `year` and `round_number` should match a race in the races dataset; `grid_position` is the hypothetical starting place.

In [ ]:
# EDIT THESE VALUES
chosen_driver = "Lewis Hamilton"
chosen_constructor = "Mercedes"
chosen_year = int(model_df["year"].max())
chosen_round = 1
chosen_grid = 5

simulation_result = simulate_race(
    driver_name=chosen_driver,
    constructor_name=chosen_constructor,
    year=chosen_year,
    round_number=chosen_round,
    grid_position=chosen_grid
)

display(pd.DataFrame([simulation_result]))

## 14. Compare different starting-grid scenarios

This table compares predicted outcomes for several hypothetical grid positions for the same driver and constructor.

In [ ]:
grid_scenarios = [1, 3, 5, 10, 15, 20]
scenario_results = []

for grid_position in grid_scenarios:
    scenario_results.append(
        simulate_race(
            driver_name=chosen_driver,
            constructor_name=chosen_constructor,
            year=chosen_year,
            round_number=chosen_round,
            grid_position=grid_position
        )
    )

scenario_table = pd.DataFrame(scenario_results)
display(scenario_table)

## 15. Export model-ready data

The merged file is useful for analysis. This second file contains the model features and target, making it easier to reuse the prepared dataset.

In [ ]:
model_export_columns = feature_columns + ["positionOrder", "driver_name", "constructor_name", "race_name", "date"]
model_df[model_export_columns].to_csv("f1_model_ready_dataset.csv", index=False)
print("Saved f1_model_ready_dataset.csv")

from google.colab import files
files.download("f1_model_ready_dataset.csv")

## How to improve this project next

To move from a historical predictor to a more realistic F1 strategy simulator, add datasets with:
- Lap-by-lap timing and position.
- Tyre compound and tyre age for each stint.
- Pit-stop lap and pit-lane duration.
- Weather and track temperature.
- Safety-car and virtual-safety-car periods.
- Fuel load or a reasonable fuel-burn estimate.

Then you can simulate one-stop versus two-stop strategies, estimate stint pace and pit-loss, and compare expected race time under different safety-car scenarios.

## 16. Create the Streamlit deployment files

Run this cell to create `app.py`, `requirements.txt`, and `README_Streamlit.txt`. The generated app reads the already-merged `f1_merged_dataset.csv` file directly. It no longer asks users to upload or merge `races.csv`, `results.csv`, `drivers.csv`, and `constructors.csv`.

**Before running the app**, keep `f1_merged_dataset.csv` in the same folder as `app.py`. The dataset is not embedded inside the app.

In [ ]:
# Create the Streamlit app using the pre-merged F1 dataset.
app_code = 'from pathlib import Path\n\nimport streamlit as st\nimport pandas as pd\nimport numpy as np\nfrom sklearn.ensemble import RandomForestRegressor, RandomForestClassifier\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.metrics import (\n    mean_absolute_error,\n    mean_squared_error,\n    accuracy_score,\n    classification_report,\n)\n\n# -------------------------------------------------------------------\n# App setup: use ONE already-merged F1 dataset\n# -------------------------------------------------------------------\nst.set_page_config(page_title="F1 Race Strategy Simulator", page_icon="🏎️", layout="wide")\nst.title("🏎️ F1 Race Strategy Simulator")\nst.caption(\n    "Uses the pre-merged F1 dataset to estimate finishing position and podium probability. "\n    "It does not optimize tyre or pit-stop strategy."\n)\n\nBASE_DIR = Path(__file__).resolve().parent\nDATA_PATH = BASE_DIR / "f1_merged_dataset.csv"\n\n\n@st.cache_data(show_spinner="Loading and preparing the merged F1 dataset...")\ndef load_merged_data(csv_path: str):\n    """Load one merged CSV and prepare historical features for the ML models."""\n    path = Path(csv_path)\n    if not path.is_file():\n        raise FileNotFoundError(\n            f"Could not find {path.name}. Place f1_merged_dataset.csv in the same folder as app.py."\n        )\n\n    df = pd.read_csv(path, na_values=[r"\\N"])\n\n    # Normalize common column-name variations from merged F1 datasets.\n    aliases = {\n        "positionOrder": ["positionOrder", "position_order", "finish_position", "position"],\n        "year": ["year", "season"],\n        "round": ["round", "race_round"],\n        "grid": ["grid", "grid_position", "starting_grid"],\n        "driver_name": ["driver_name", "driverName", "full_name", "driver"],\n        "constructor_name": ["constructor_name", "constructorName", "team_name", "constructor", "team"],\n        "driverId": ["driverId", "driver_id"],\n        "constructorId": ["constructorId", "constructor_id", "teamId", "team_id"],\n        "raceId": ["raceId", "race_id"],\n        "circuitId": ["circuitId", "circuit_id"],\n        "points": ["points", "driver_points"],\n        "date": ["date", "race_date"],\n        "race_name": ["race_name", "raceName", "name", "race"],\n    }\n    for standard_name, candidates in aliases.items():\n        if standard_name not in df.columns:\n            found = next((col for col in candidates if col in df.columns), None)\n            if found is not None:\n                df = df.rename(columns={found: standard_name})\n\n    # Some merged files store driver names as separate forename/surname columns.\n    if "driver_name" not in df.columns and {"forename", "surname"}.issubset(df.columns):\n        df["driver_name"] = (\n            df["forename"].fillna("").astype(str) + " " + df["surname"].fillna("").astype(str)\n        ).str.strip()\n\n    required = ["year", "round", "grid", "positionOrder", "driver_name", "constructor_name"]\n    missing = [column for column in required if column not in df.columns]\n    if missing:\n        raise ValueError(\n            "The merged CSV is missing required columns: " + ", ".join(missing)\n            + ". Check that you are using f1_merged_dataset.csv created after merging races, results, drivers, and constructors."\n        )\n\n    # IDs are helpful model features; create stable numeric IDs if the merged file lacks them.\n    if "driverId" not in df.columns:\n        df["driverId"] = pd.factorize(df["driver_name"].astype(str))[0] + 1\n    if "constructorId" not in df.columns:\n        df["constructorId"] = pd.factorize(df["constructor_name"].astype(str))[0] + 1\n    if "raceId" not in df.columns:\n        df["raceId"] = pd.factorize(\n            df["year"].astype(str) + "_" + df["round"].astype(str)\n        )[0] + 1\n    if "circuitId" not in df.columns:\n        df["circuitId"] = 0\n    if "points" not in df.columns:\n        df["points"] = 0\n    if "date" not in df.columns:\n        df["date"] = pd.NaT\n\n    numeric_columns = [\n        "year", "round", "grid", "positionOrder", "points", "raceId",\n        "driverId", "constructorId", "circuitId",\n    ]\n    for column in numeric_columns:\n        df[column] = pd.to_numeric(df[column], errors="coerce")\n    df["date"] = pd.to_datetime(df["date"], errors="coerce")\n    df = df.sort_values(["year", "round", "raceId", "positionOrder"]).reset_index(drop=True)\n\n    model_df = df.dropna(subset=["year", "round", "grid", "positionOrder"]).copy()\n    model_df = model_df[model_df["grid"] > 0].copy()\n    model_df = model_df.sort_values(["year", "round", "raceId", "positionOrder"]).reset_index(drop=True)\n\n    # Historical features only use earlier races, avoiding use of the current result as an input.\n    model_df["driver_previous_finish"] = model_df.groupby("driverId")["positionOrder"].transform(lambda s: s.shift(1))\n    model_df["driver_avg_finish_before"] = model_df.groupby("driverId")["positionOrder"].transform(lambda s: s.shift(1).expanding().mean())\n    model_df["driver_avg_points_before"] = model_df.groupby("driverId")["points"].transform(lambda s: s.shift(1).expanding().mean())\n    model_df["team_previous_finish"] = model_df.groupby("constructorId")["positionOrder"].transform(lambda s: s.shift(1))\n    model_df["team_avg_finish_before"] = model_df.groupby("constructorId")["positionOrder"].transform(lambda s: s.shift(1).expanding().mean())\n    model_df["team_avg_points_before"] = model_df.groupby("constructorId")["points"].transform(lambda s: s.shift(1).expanding().mean())\n\n    features = [\n        "year", "round", "circuitId", "grid", "driverId", "constructorId",\n        "driver_previous_finish", "driver_avg_finish_before", "driver_avg_points_before",\n        "team_previous_finish", "team_avg_finish_before", "team_avg_points_before",\n    ]\n    races = model_df[["year", "round", "circuitId"]].drop_duplicates().copy()\n    return df, races, model_df, features\n\n\ntry:\n    df, races, model_df, feature_columns = load_merged_data(str(DATA_PATH))\nexcept Exception as exc:\n    st.error(f"Could not load the merged F1 dataset. Details: {exc}")\n    st.markdown(\n        "**Required project structure**\\n\\n"\n        "```text\\nF1_Race_Strategy_Simulator/\\n"\n        "├── app.py\\n"\n        "└── f1_merged_dataset.csv\\n```"\n    )\n    st.stop()\n\nst.success(f"Loaded {len(df):,} rows from f1_merged_dataset.csv.")\nyears = sorted(model_df["year"].dropna().astype(int).unique())\nif len(years) < 2:\n    st.error("At least two different seasons are needed to train and test the model.")\n    st.stop()\n\nsplit_index = min(max(1, int(len(years) * 0.8)), len(years) - 1)\ntest_start_year = years[split_index]\ntrain_data = model_df[model_df["year"] < test_start_year].copy()\ntest_data = model_df[model_df["year"] >= test_start_year].copy()\nif train_data.empty or test_data.empty:\n    st.error("Not enough data after the chronological train/test split.")\n    st.stop()\n\n\n@st.cache_resource\ndef train_models(train_x, train_y, train_podium_y):\n    reg = Pipeline([\n        ("imputer", SimpleImputer(strategy="median")),\n        ("model", RandomForestRegressor(n_estimators=250, min_samples_leaf=3, random_state=42, n_jobs=-1)),\n    ])\n    clf = Pipeline([\n        ("imputer", SimpleImputer(strategy="median")),\n        ("model", RandomForestClassifier(n_estimators=250, min_samples_leaf=3, class_weight="balanced", random_state=42, n_jobs=-1)),\n    ])\n    reg.fit(train_x, train_y)\n    clf.fit(train_x, train_podium_y)\n    return reg, clf\n\n\nwith st.spinner("Training the models on the merged historical dataset..."):\n    train_podium = train_data["positionOrder"].between(1, 3).astype(int)\n    regressor, podium_model = train_models(train_data[feature_columns], train_data["positionOrder"], train_podium)\n\ntest_pred = regressor.predict(test_data[feature_columns])\ntest_podium_true = test_data["positionOrder"].between(1, 3).astype(int)\ntest_podium_pred = podium_model.predict(test_data[feature_columns])\nmae = mean_absolute_error(test_data["positionOrder"], test_pred)\nrmse = np.sqrt(mean_squared_error(test_data["positionOrder"], test_pred))\naccuracy = accuracy_score(test_podium_true, test_podium_pred)\n\nst.subheader("Model evaluation")\nm1, m2, m3, m4 = st.columns(4)\nm1.metric("Test seasons", f"{int(test_data[\'year\'].min())}–{int(test_data[\'year\'].max())}")\nm2.metric("Finishing-position MAE", f"{mae:.2f} places")\nm3.metric("Finishing-position RMSE", f"{rmse:.2f} places")\nm4.metric("Podium classification accuracy", f"{accuracy:.1%}")\nwith st.expander("View classification report"):\n    report = classification_report(test_podium_true, test_podium_pred, labels=[0, 1], target_names=["Not podium", "Podium"], zero_division=0, output_dict=True)\n    st.dataframe(pd.DataFrame(report).transpose().round(3), use_container_width=True)\n    st.caption("Accuracy can be misleading when podium results are less common. Check podium precision, recall, and F1 too.")\n\nst.divider()\nst.subheader("Predict a hypothetical race")\ndrivers = sorted(df["driver_name"].dropna().astype(str).loc[lambda s: s.str.strip().ne("")].unique().tolist())\nconstructors = sorted(df["constructor_name"].dropna().astype(str).unique().tolist())\nif not drivers or not constructors:\n    st.error("No driver or constructor names were found in the merged dataset.")\n    st.stop()\n\nc1, c2, c3 = st.columns(3)\nwith c1:\n    driver_name = st.selectbox("Driver", drivers)\nwith c2:\n    constructor_name = st.selectbox("Constructor / team", constructors)\nwith c3:\n    prediction_year = st.selectbox("Season", list(range(int(max(years)), int(min(years)) - 1, -1)))\n\navailable_rounds = sorted(races.loc[races["year"] == prediction_year, "round"].dropna().astype(int).unique().tolist())\nif not available_rounds:\n    available_rounds = sorted(model_df["round"].dropna().astype(int).unique().tolist())\nround_number = st.selectbox("Round", available_rounds, index=0)\ngrid_position = st.slider("Hypothetical starting grid position", min_value=1, max_value=20, value=5)\n\n\ndef simulate(driver, team, year, rnd, grid):\n    earlier = model_df[(model_df["year"] < year) | ((model_df["year"] == year) & (model_df["round"] < rnd))]\n    dh = earlier[earlier["driver_name"].astype(str).str.casefold() == driver.casefold()]\n    th = earlier[earlier["constructor_name"].astype(str).str.casefold() == team.casefold()]\n    race_match = races[(races["year"] == year) & (races["round"] == rnd)]\n    circuit_id = race_match["circuitId"].iloc[0] if len(race_match) else np.nan\n    driver_rows = df[df["driver_name"].astype(str).str.casefold() == driver.casefold()]\n    team_rows = df[df["constructor_name"].astype(str).str.casefold() == team.casefold()]\n    row = {\n        "year": year,\n        "round": rnd,\n        "circuitId": circuit_id,\n        "grid": grid,\n        "driverId": driver_rows["driverId"].iloc[0] if len(driver_rows) else np.nan,\n        "constructorId": team_rows["constructorId"].iloc[0] if len(team_rows) else np.nan,\n        "driver_previous_finish": dh["positionOrder"].iloc[-1] if len(dh) else np.nan,\n        "driver_avg_finish_before": dh["positionOrder"].mean() if len(dh) else np.nan,\n        "driver_avg_points_before": dh["points"].mean() if len(dh) else np.nan,\n        "team_previous_finish": th["positionOrder"].iloc[-1] if len(th) else np.nan,\n        "team_avg_finish_before": th["positionOrder"].mean() if len(th) else np.nan,\n        "team_avg_points_before": th["points"].mean() if len(th) else np.nan,\n    }\n    X = pd.DataFrame([row])[feature_columns]\n    predicted_position = float(regressor.predict(X)[0])\n    probabilities = podium_model.predict_proba(X)[0]\n    podium_class_index = list(podium_model.classes_).index(1) if 1 in podium_model.classes_ else 0\n    podium_probability = float(probabilities[podium_class_index])\n    return predicted_position, podium_probability\n\n\nif st.button("Run prediction", type="primary"):\n    predicted_position, podium_probability = simulate(driver_name, constructor_name, int(prediction_year), int(round_number), int(grid_position))\n    a, b = st.columns(2)\n    a.metric("Estimated finishing position", f"{predicted_position:.1f}")\n    b.metric("Estimated podium probability", f"{podium_probability:.1%}")\n    st.caption("These are estimates based on historical results, not guarantees. This version does not model tyre wear, pit stops, weather, or safety cars.")\n\nst.subheader("Compare starting-grid scenarios")\nif st.button("Compare grid positions"):\n    rows = []\n    for grid in [1, 3, 5, 10, 15, 20]:\n        pos, prob = simulate(driver_name, constructor_name, int(prediction_year), int(round_number), grid)\n        rows.append({"Grid position": grid, "Estimated finishing position": round(pos, 1), "Podium probability": f"{prob:.1%}"})\n    st.dataframe(pd.DataFrame(rows), use_container_width=True)\n'

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

requirements_text = 'streamlit\npandas\nnumpy\nscikit-learn\n'
with open('requirements.txt', 'w', encoding='utf-8') as f:
    f.write(requirements_text)

readme_text = '''# F1 Race Strategy Simulator\n\n## Files required\n- `app.py`\n- `f1_merged_dataset.csv` (the merged dataset created earlier in this notebook)\n- `requirements.txt`\n\n## Run locally\nInstall dependencies with `pip install -r requirements.txt`, then run `streamlit run app.py`.\n\nThe app trains its prediction models from the merged historical data when it starts. It estimates finishing position and podium probability; it does not model tyre wear, pit stops, weather, or safety cars.\n'''
with open('README_Streamlit.txt', 'w', encoding='utf-8') as f:
    f.write(readme_text)

print('Created app.py, requirements.txt, and README_Streamlit.txt')

### Download the deployment bundle
Run the next cell in Google Colab to download a ZIP containing `app.py`, `requirements.txt`, and a short README. Keep `f1_merged_dataset.csv` alongside `app.py` when running or deploying the app.

In [ ]:
import zipfile
from google.colab import files

with zipfile.ZipFile('F1_Streamlit_Deployment_Bundle.zip', 'w', zipfile.ZIP_DEFLATED) as z: 
    z.write('app.py')
    z.write('requirements.txt')
    z.write('README_Streamlit.txt')

files.download('F1_Streamlit_Deployment_Bundle.zip')
